[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/6_optimizacion/_assets/03_reactor_continuo.ipynb)

# Notebook 3 · El reactor

Acompaña a la **clase 3** de la unidad de modelado y optimización. Da por leídas
las cinco páginas.

Las páginas calculan a mano y dibujan tres figuras. Aquí la computadora rehace
las dos cosas desde cero: resuelve el reparto con `scipy`, **construye** los
multiplicadores en vez de citarlos, y dibuja lo que las páginas solo pueden
tabular —el valor óptimo contra la potencia, y las tres trayectorias del
descenso—.

Lo que **no** hace es comprobar la estacionariedad después de haber despejado los
multiplicadores de la estacionariedad: eso quedaría satisfecho por construcción y
no comprobaría nada. Lo que hace es construirlos y verificar **las otras tres**
condiciones.

In [ ]:
# === Celda 1 · Preparación =====================================
# La convención de signos, otra vez, porque es la fuente número uno de errores al
# pasar del papel al código:
#
#   - En las páginas el problema del reactor es de MÁXIMO.
#   - scipy MINIMIZA siempre. Un máximo se resuelve minimizando -f, y al valor
#     que devuelve hay que cambiarle el signo otra vez.
#   - El lagrangeano de las páginas lleva los términos RESTANDO:
#         L = f - lambda*(h - c) - mu*(g - b)
#     y con esa convención cada multiplicador ES la derivada del valor óptimo
#     respecto de su lado derecho. La forma estándar de los libros (mínimo, con
#     g <= 0 y L = f + mu*g) devuelve el mismo número con el signo cambiado.

import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

B = np.array([6.0, 8.0, 10.0])      # lo que paga la primera unidad de cada sistema
P_TOTAL = 15.0                      # potencia por turno
SISTEMAS = ['escudos', 'motores', 'soporte vital']

def rendimiento(p):
    """u(p) = sum_i (b_i p_i - p_i^2 / 2). Cóncava: cada unidad rinde uno menos."""
    return float(np.sum(B * p - p**2 / 2))

def negativo(p):
    return -rendimiento(p)

print('rendimiento del reparto parejo (5,5,5):', rendimiento(np.array([5., 5., 5.])))
print('rendimiento de (3,5,7):               ', rendimiento(np.array([3., 5., 7.])))

## 1 · El reparto óptimo, con la igualdad

`minimize` con una restricción de igualdad. El punto inicial es el reparto
parejo, que es factible y no es óptimo.

In [ ]:
# === Celda 2 · Resolver el reparto =============================
igualdad = {'type': 'eq', 'fun': lambda p: np.sum(p) - P_TOTAL}
cotas_no_negativas = [(0, None)] * 3

sol = minimize(negativo, x0=np.array([5., 5., 5.]),
               constraints=[igualdad], bounds=cotas_no_negativas)

print('reparto óptimo :', np.round(sol.x, 6))
print('rendimiento    :', round(-sol.fun, 6))
print('¿suma 15?      :', round(float(np.sum(sol.x)), 9))
print()
print('la página dice (3, 5, 7) y 86.5')
assert np.allclose(sol.x, [3, 5, 7], atol=1e-5)
assert abs(-sol.fun - 86.5) < 1e-6

## 2 · El gradiente, a mano y numérico

La página 2 calcula $\partial f/\partial p_i = b_i - p_i$ derivando. Aquí se
comprueba contra una diferencia central, que no sabe nada de cálculo simbólico.

In [ ]:
# === Celda 3 · Gradiente propio contra numérico ================
def gradiente(p):
    return B - p                       # derivada de b_i p_i - p_i^2/2

def gradiente_numerico(f, p, h=1e-6):
    salida = np.zeros_like(p)
    for i in range(len(p)):
        paso = np.zeros_like(p); paso[i] = h
        salida[i] = (f(p + paso) - f(p - paso)) / (2 * h)
    return salida

for p in (np.array([5., 5., 5.]), sol.x):
    a, b = gradiente(p), gradiente_numerico(rendimiento, p)
    print(f'en {np.round(p, 3)}:  a mano {np.round(a, 6)}   numérico {np.round(b, 6)}')

print()
print('en el óptimo las tres componentes son iguales: ese número común es lambda')

## 3 · Construir $\lambda$, y verificar las otras tres condiciones

El método de `minimize` que se usa arriba **no devuelve multiplicadores**. Y
despejarlos de la estacionariedad para después «verificar» la estacionariedad
sería circular.

Así que se **construye** $\lambda$ desde la estacionariedad —las restricciones son
afines, así que $\lambda = b_i - p_i$— y se verifican las otras tres condiciones
KKT: factibilidad, signo y holgura complementaria.

In [ ]:
# === Celda 4 · El multiplicador de la igualdad =================
lam = float(np.mean(B - sol.x))          # construido, no verificado
print('lambda construido :', round(lam, 6))
print('¿los tres coinciden? ', np.round(B - sol.x, 6))

# Verificación de las OTRAS condiciones (la estacionariedad se usó para construir)
print()
print('factibilidad  suma = 15 :', abs(float(np.sum(sol.x)) - P_TOTAL) < 1e-6)
print('factibilidad  p >= 0    :', bool(np.all(sol.x > -1e-9)))
print('signo         lambda es libre en una igualdad: aquí sale', round(lam, 3), '> 0')
print('holgura       no aplica: una igualdad está siempre activa')

## 4 · $\lambda$ es una derivada, no una diferencia

Aquí está la figura que la página solo puede tabular. Se resuelve el problema
para muchos valores de la potencia total, se dibuja $U^*(P)$, y encima se
dibujan dos cosas:

- la **recta tangente** en $P=15$, cuya pendiente es $\lambda = 3$;
- la **secante** que une $P=15$ con $P=16$, cuya pendiente es la diferencia
  finita, $17/6 = 2.833\ldots$

Que las dos rectas no coincidan **es** la lección de la página 3.

In [ ]:
# === Celda 5 · El valor óptimo contra la potencia ==============
def optimo(P, cota=None):
    """Devuelve (reparto, valor) resolviendo el problema con potencia P."""
    restr = [{'type': 'eq', 'fun': lambda p, P=P: np.sum(p) - P}]
    topes = [(0, None), (0, None), (0, cota)]
    r = minimize(negativo, x0=np.full(3, P / 3), constraints=restr, bounds=topes)
    return r.x, -r.fun

Ps = np.linspace(6, 22, 120)
valores = np.array([optimo(P)[1] for P in Ps])

U15 = optimo(15.0)[1]
U16 = optimo(16.0)[1]
print('U*(15) =', round(U15, 6), '   U*(16) =', round(U16, 6))
print('diferencia finita :', round(U16 - U15, 6), ' (17/6 = 2.8333...)')
print('derivada (lambda) :', round(lam, 6))
print('forma cerrada     :', round(-15**2 / 6 + 8 * 15 + 4, 6), '= U*(15)')

In [ ]:
# === Celda 6 · La figura: tangente contra secante ==============
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(Ps, valores, lw=2.5, label='$U^*(P)$, resuelto con scipy')

recta_tangente = U15 + lam * (Ps - 15)
ax.plot(Ps, recta_tangente, '--', lw=2,
        label=f'tangente en P=15, pendiente $\\lambda$ = {lam:.3f}')

pendiente_secante = U16 - U15
ax.plot(Ps, U15 + pendiente_secante * (Ps - 15), ':', lw=2,
        label=f'secante 15 a 16, pendiente = {pendiente_secante:.3f}')

ax.plot([15, 16], [U15, U16], 'o', ms=7, color='black', zorder=5)
ax.annotate('P = 15', (15, U15), textcoords='offset points', xytext=(-42, 6))
ax.annotate('P = 16', (16, U16), textcoords='offset points', xytext=(6, -14))
ax.set_xlim(11, 20); ax.set_ylim(min(valores[Ps > 11]) - 2, max(valores) + 4)
ax.set_xlabel('potencia total disponible, P')
ax.set_ylabel('rendimiento del mejor reparto')
ax.set_title('El multiplicador es la pendiente, no el escalón')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=.25)
plt.show()

**Fíjate dónde se separan.** Cerca de $P=15$ las dos rectas casi coinciden: por
eso en un problema lineal, donde $U^*$ es **recta** dentro del rango de validez,
la derivada y la diferencia dan lo mismo y nadie nota la distinción. En cuanto la
curva se dobla, la unidad completa rinde menos que su primera fracción.

Y el rango de la forma cerrada, comprobado por los dos lados:

In [ ]:
# === Celda 7 · Dónde deja de valer la forma cerrada ============
cerrada = lambda P: -P**2 / 6 + 8 * P + 4

print(f'{"P":>4} {"scipy":>12} {"forma cerrada":>15}   ¿coinciden?')
for P in (3.0, 5.0, 6.0, 9.0, 15.0):
    real = optimo(P)[1]
    print(f'{P:>4.0f} {real:>12.4f} {cerrada(P):>15.4f}   {abs(real - cerrada(P)) < 1e-4}')
print()
print('por debajo de P = 6 el óptimo deja de darle potencia a los escudos')
print('reparto en P = 3:', np.round(optimo(3.0)[0], 4))

## 5 · La cota del soporte vital, y el signo de $\mu$

Ahora la desigualdad $p_3 \le 5$. Se construye $\mu_3$ desde la estacionariedad
del sistema topado y se verifican las otras tres condiciones, incluido el signo,
que en un **máximo con $\le$** tiene que ser $\mu_3 \ge 0$.

In [ ]:
# === Celda 8 · KKT con la cota =================================
p_cota, U_cota = optimo(15.0, cota=5.0)
lam_c = float(np.mean((B - p_cota)[:2]))       # de los dos sistemas libres
mu3 = float((B - p_cota)[2] - lam_c)           # del sistema topado

print('reparto con cota :', np.round(p_cota, 6))
print('rendimiento      :', round(U_cota, 6), '  (sin cota era', round(U15, 4), ')')
print('lambda           :', round(lam_c, 6))
print('mu_3             :', round(mu3, 6))
print()
print('--- las otras tres condiciones ---')
print('factibilidad   suma 15 y p3 <= 5 :',
      abs(float(np.sum(p_cota)) - 15) < 1e-6 and p_cota[2] <= 5 + 1e-9)
print('signo          mu_3 >= 0         :', mu3 > -1e-9,
      '(máximo con <=: aflojar no puede empeorar)')
print('holgura        mu_3 * (p3 - 5)   :', round(mu3 * (p_cota[2] - 5), 9),
      '(la cota está activa, así que puede empujar)')

In [ ]:
# === Celda 9 · Y los dos multiplicadores, comprobados como derivadas ===
# mu_3 tiene que ser lo que rinde una unidad MÁS de cota, y lambda lo que rinde
# una unidad más de potencia. Los dos se comprueban moviendo su lado derecho.
h = 1e-5
d_cota = (optimo(15.0, cota=5.0 + h)[1] - optimo(15.0, cota=5.0 - h)[1]) / (2 * h)
d_pot  = (optimo(15.0 + h, cota=5.0)[1] - optimo(15.0 - h, cota=5.0)[1]) / (2 * h)

print(f'mu_3   construido {mu3:.4f}   derivada numérica {d_cota:.4f}')
print(f'lambda construido {lam_c:.4f}   derivada numérica {d_pot:.4f}')
print()
print('lambda baja de 3 a 2: con el mejor sistema tapado, la potencia extra vale menos')
print('la cota cuesta:', round(U15 - U_cota, 4))

## 6 · Los cuatro signos, resueltos en vez de citados

La tabla de la página 4 dice que el signo del multiplicador sale de dos
preguntas: si subir $b$ afloja o aprieta, y si el problema maximiza o minimiza.

Aquí se comprueba en el caso más pequeño posible —una parábola y una cota— sin
usar ninguna regla: se **deriva el valor óptimo** y se mira qué signo tiene.

In [ ]:
# === Celda 10 · La tabla de signos, calculada ==================
# opt (x-4)^2 sujeto a x <= b o x >= b. En los cuatro casos la cota está activa.
def valor_optimo(sentido, direccion, b):
    f = (lambda x: -(x - 4)**2) if sentido == 'max' else (lambda x: (x - 4)**2)
    libre = 4.0
    factible = (libre <= b) if direccion == '<=' else (libre >= b)
    x = libre if factible else b
    return x, f(x)

h = 1e-6
print(f'{"problema":>9} {"restricción":>12} {"subir b":>9} {"mu = df*/db":>13}   signo')
for sentido in ('max', 'min'):
    for direccion in ('<=', '>='):
        b = 2.0 if direccion == '<=' else 6.0
        _, f0 = valor_optimo(sentido, direccion, b)
        _, f1 = valor_optimo(sentido, direccion, b + h)
        mu = (f1 - f0) / h
        afloja = 'afloja' if direccion == '<=' else 'aprieta'
        print(f'{sentido:>9} {("g(x) " + direccion + " b"):>12} {afloja:>9} '
              f'{mu:>13.3f}   {"mu >= 0" if mu > 0 else "mu <= 0"}')

Los cuatro renglones salen de derivar, no de recordar. Y el puente con los
libros, que usan la forma estándar (mínimo, $g(x) \le 0$, y el término
**sumando**):

In [ ]:
# === Celda 11 · El multiplicador del libro es menos el de aquí ==
# min (x-4)^2 sujeto a x <= 2, que en forma estándar es g(x) = x - 2 <= 0.
x_opt = 2.0
mu_notas = 2 * (x_opt - 4)      # de  L = f - mu*(g - b),  grad f = mu * grad g
mu_libro = -mu_notas            # de  L = f + mu*g

print('multiplicador en la convención de estas notas :', mu_notas)
print('multiplicador en la forma estándar del libro  :', mu_libro)
print()
print('el del libro sale >= 0 siempre, porque el signo ya se absorbió al')
print('escribir la forma estándar. Es el mismo número, no otro resultado.')

## 7 · Bajar la pendiente

El último método de la clase, sobre el problema de la bomba:
$f(x,y) = (x-3)^2 + 4(y-2)^2$, sin restricciones. Se implementa en seis líneas y
se corre con los tres tamaños de paso de la página 5.

In [ ]:
# === Celda 12 · El descenso, y los tres tamaños de paso ========
def desgaste(v):
    x, y = v
    return (x - 3)**2 + 4 * (y - 2)**2

def grad_desgaste(v):
    x, y = v
    return np.array([2 * (x - 3), 8 * (y - 2)])

def descenso(alpha, x0=np.array([0., 0.]), pasos=40, tol=1e-9):
    camino = [x0.copy()]
    x = x0.copy()
    for _ in range(pasos):
        g = grad_desgaste(x)
        if np.linalg.norm(g) <= tol:
            break
        x = x - alpha * g
        camino.append(x.copy())
    return np.array(camino)

for alpha in (0.1, 0.25, 0.3):
    c = descenso(alpha, pasos=3)
    print(f'alpha = {alpha}:  ' + '  ->  '.join(np.array2string(p, precision=3) for p in c))
print()
print('la página tabula exactamente la primera fila: (0.6,1.6), (1.08,1.92), (1.464,1.984)')

In [ ]:
# === Celda 13 · Las tres trayectorias, dibujadas ===============
fig, ejes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True)
malla_x, malla_y = np.meshgrid(np.linspace(-1.5, 6, 300), np.linspace(-3, 6.5, 300))
Z = (malla_x - 3)**2 + 4 * (malla_y - 2)**2

# Menos pasos en el que diverge: con doce, la trayectoria se sale a mil millones
# y el panel queda como una barra vertical maciza en la que no se ve nada.
for ax, (alpha, nota, pasos) in zip(ejes, ((0.1, 'converge', 12),
                                           (0.25, 'la y salta para siempre', 8),
                                           (0.3, 'diverge', 3))):
    ax.contour(malla_x, malla_y, Z, levels=[1, 4, 9, 16, 25, 36], linewidths=.8, alpha=.6)
    camino = descenso(alpha, pasos=pasos)
    ax.plot(camino[:, 0], camino[:, 1], 'o-', ms=4, lw=1.6)
    ax.plot(3, 2, '*', ms=14)
    ax.set_title(f'$\\alpha$ = {alpha} — {nota}')
    ax.set_xlim(-1.5, 6); ax.set_ylim(-3, 6.5)
    ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

**Cambia el `alpha` de la tercera y busca el umbral tú.** Lo que falta en cada
coordenada se multiplica, en cada paso, por $1-2\alpha$ en $x$ y por $1-8\alpha$
en $y$. El método converge exactamente mientras los dos factores estén entre
$-1$ y $1$, y el que manda es el segundo: el umbral está en $\alpha = 1/4$.

In [ ]:
# === Celda 14 · El umbral, medido y no derivado ================
print(f'{"alpha":>7} {"factor en x":>12} {"factor en y":>12} {"lo que falta tras 60 pasos":>28}')
for alpha in (0.1, 0.2, 0.24, 0.25, 0.26, 0.3):
    camino = descenso(alpha, pasos=60)
    falta = np.abs(camino[-1] - np.array([3., 2.]))
    print(f'{alpha:>7.2f} {1 - 2 * alpha:>12.2f} {1 - 8 * alpha:>12.2f} '
          f'{np.array2string(falta, precision=3, max_line_width=99):>28}')
print()
print('con 0.24 converge, pero despacio: que tarde no es que diverja.')
print('con 0.25 la coordenada y se queda clavada a distancia 2, y la x sí converge.')